# 🎙️ speakerscribe v0.2.0 — Production Notebook

**Automatic transcription + speaker diarization with smart chunking for long audio files.**

> This notebook runs the full `speakerscribe` pipeline: it converts audio/video files to text, identifies who is speaking at each moment (diarization), and organizes the output in multiple formats ready for human reading or LLM ingestion.

Powered by [faster-whisper](https://github.com/SYSTRAN/faster-whisper) (OpenAI Whisper, optimized) + [pyannote.audio](https://github.com/pyannote/pyannote-audio) (speaker diarization).

---

## ✅ Before you start — checklist

Complete these **3 mandatory steps** before running any cell:

1. **Enable GPU runtime**: `Runtime → Change runtime type → T4 GPU`  
   *(Transcription with Whisper is ~10x faster on GPU than CPU)*

2. **HuggingFace token (Read access)** stored in Colab Secrets as `HF_TOKEN`:
   - Create a free token at https://huggingface.co/settings/tokens  
     *(Select token type **Read**, NOT fine-grained)*
   - Accept the model terms at https://huggingface.co/pyannote/speaker-diarization-community-1  
     *(Required to use the diarization model)*
   - In Colab: click the 🔑 icon in the left sidebar → add secret `HF_TOKEN`

3. **Place your audio/video files** inside `<WORKSPACE>/data/`  
   Supported formats: `.mp3`, `.mp4`, `.wav`, `.m4a`, `.ogg`, `.flac`, `.webm`

4. *(Optional)* **Per-file glossary**: create a `<audio_filename>.prompt.txt` file next to each audio file with proper nouns, names, and technical terms to improve transcription accuracy.

---

## 🆕 What's new in v0.2

| Feature | Description |
|---|---|
| ✂️ **Auto chunking** | Audio files > 2h are automatically split into 30-min chunks with 5s overlap |
| 🧠 **Full-audio diarization** | Speaker detection always runs on the complete audio (better speaker consistency) |
| 🔁 **Param-aware diarization cache** | Changing `num_speakers` automatically invalidates the cache |
| 📊 **Per-stage telemetry** | Each `.json` output includes timing breakdowns per pipeline stage |
| 🚫 **Filler word filter** | Removes `uh`, `um`, `like`... from `.transcript.md` output |
| 📚 **Per-file glossary** | Use `<stem>.prompt.txt` to apply custom vocabulary per audio file |
| 📄 **Unified LLM output** | `_full_for_llm.txt` consolidates everything for long-context LLMs |
| 💾 **SQLite history** | Content-hash-based idempotency — reprocessing is safe and skips completed files |
| 📈 **Batch summary** | Generates `entregables/_resumen.md` with stats for all processed files |

---
## Step 1 — Mount Google Drive

This mounts your Google Drive at `/content/drive/`, making all your files accessible from this notebook.

**You must run this cell every time you start a new session.** After mounting, you'll be prompted to authorize access — click the link and paste the code.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

---
## Step 2 — Configuration (the only cell you need to edit)

> 💡 **Edit ONLY this cell.** All other cells reuse these variables without changes.

### Key parameters explained:

| Parameter | What it does |
|---|---|
| `WORKSPACE` | Root folder path in your Drive. Must contain a `data/` subfolder with your audio files. |
| `MODEL` | Whisper model size. `large-v3-turbo` is the recommended sweet spot of speed vs quality. |
| `LANGUAGE` | Audio language code (`"es"`, `"en"`, `"pt"`, etc.) or `None` for auto-detection. |
| `ENABLE_DIARIZATION` | Whether to identify and label different speakers. Set `False` if you only need raw text. |
| `NUM_SPEAKERS` | Set this if you know the exact number of speakers. Leave `None` to auto-detect within `MIN/MAX`. |
| `INITIAL_PROMPT` | Global vocabulary hint for Whisper — list proper nouns, names, or jargon here for better accuracy. |
| `LONG_AUDIO_THRESHOLD_MIN` | Audio files longer than this (in minutes) will be automatically chunked. |
| `FORCE_REPROCESS` | Set `True` to reprocess all files regardless of existing outputs (ignores cache). |

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDIT HERE — this is the only cell that requires changes
# ══════════════════════════════════════════════════════════════════

# ── Workspace (root folder in Drive, must contain a data/ subfolder) ──
WORKSPACE = "/content/drive/MyDrive/Tests/Speakerscribe"

# ── Whisper model ─────────────────────────────────────────────────
# 'tiny'           ~1.5 GB VRAM  — for smoke tests only
# 'small'          ~2.5 GB VRAM  — fast, decent quality
# 'large-v3-turbo' ~3.5 GB VRAM  — RECOMMENDED: fast + excellent accuracy ⭐
# 'large-v3'       ~5.0 GB VRAM  — maximum quality, slower
MODEL = "large-v3-turbo"

# ── Language (None = auto-detect) ─────────────────────────────────
LANGUAGE = "en"   # 'es', 'en', 'pt', 'fr'... or None for auto-detection

# ── Diarization (speaker identification) ──────────────────────────
ENABLE_DIARIZATION = True
NUM_SPEAKERS = None   # If you KNOW the exact count -> set it (e.g., 2). Otherwise leave None.
MIN_SPEAKERS = 2      # Only used when NUM_SPEAKERS is None
MAX_SPEAKERS = 8      # For typical meetings: 6 works well. 1-on-1: set NUM_SPEAKERS=2.

# ── Global vocabulary hint (HUGE quality boost for proper nouns) ───
# Applied to ALL audio files. For a specific file: create <audio_name>.prompt.txt next to it.
# Example: "ProColombia, Bancoldex, John Smith, AI analytics"
INITIAL_PROMPT = ""

# ── Long audio chunking ────────────────────────────────────────────
LONG_AUDIO_THRESHOLD_MIN = 120   # Audio files > 120 min will be auto-split
CHUNK_DURATION_MIN       = 30    # Each chunk will be 30 minutes long
CHUNK_OVERLAP_S          = 5     # 5-second overlap between consecutive chunks (prevents cut-off words)

# ── Voice Activity Detection (VAD) and transcription ──────────────
VAD_MIN_SILENCE_MS = 1000   # Silence >= N ms breaks a segment
GAP_MAX_S          = 3.0    # If the same speaker pauses > N sec, a new block is opened in .md
REMOVE_FILLERS     = True   # Removes 'uh', 'um', 'like', etc. from .transcript.md

# ── Reprocessing ──────────────────────────────────────────────────
FORCE_REPROCESS = False   # True = ignore existing outputs and redo everything

# ── Output selection (what to copy to the deliverables/ folder) ───
# The library always produces all file types in transcripts/ and splits/.
# Here you control what gets copied to entregables/ — your final consumption folder.
COPY_TRANSCRIPT_MD = True    # ⭐ Recommended: human-readable Markdown
COPY_FULL_FOR_LLM  = True    # ⭐ Feed the full transcript to Claude/GPT
COPY_SPLITS        = False   # Useful only if your LLM has a small context window
COPY_JSON          = False   # Only if you need detailed timestamps/metadata
COPY_SRT           = False   # Only if you need subtitles for video

# ══════════════════════════════════════════════════════════════════
# Summary print — verify your settings before proceeding
print(f"📂 Workspace  : {WORKSPACE}")
print(f"🧠 Model      : {MODEL}")
print(f"🌐 Language   : {LANGUAGE or 'auto-detect'}")
print(f"👥 Speakers   : {NUM_SPEAKERS or f'{MIN_SPEAKERS}-{MAX_SPEAKERS}'}")
print(f"📝 Glossary   : {(INITIAL_PROMPT[:60] + '...') if len(INITIAL_PROMPT) > 60 else (INITIAL_PROMPT or '(empty)')}")
print(f"✂️  Chunking  : audio > {LONG_AUDIO_THRESHOLD_MIN} min -> chunks of {CHUNK_DURATION_MIN} min ({CHUNK_OVERLAP_S}s overlap)")
print(f"🚫 Fillers    : {'removed from MD' if REMOVE_FILLERS else 'kept'}")
print(f"📦 Deliverables: md={COPY_TRANSCRIPT_MD}, full_llm={COPY_FULL_FOR_LLM}, splits={COPY_SPLITS}, json={COPY_JSON}, srt={COPY_SRT}")

---
## Step 3 — Install speakerscribe

> ⚠️ **After installing for the first time**: go to `Runtime → Restart runtime`, then start again from Step 1. This is a Google Colab requirement — new packages are not available until the Python kernel restarts.

Two installation options are provided:
- **Option A (Development)**: installs directly from a local copy in your Drive in editable mode — useful if you are modifying the source code.
- **Option B (Production)**: installs the latest stable version from PyPI — recommended for standard use.

In [ ]:
%%time
# ── Option A: install from Drive in editable mode (DEVELOPMENT) ───
# Use this if you have a local copy of the speakerscribe package in your Drive.
import subprocess, sys
from pathlib import Path

DRIVE_PKG = WORKSPACE  # assumes the package is in the same root folder

if (Path(DRIVE_PKG) / "pyproject.toml").exists():
    print(f"📦 Installing from {DRIVE_PKG} (editable mode)...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", DRIVE_PKG, "--quiet"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("✅ speakerscribe installed in editable mode")
        print("⚠️  If this is the first install: Runtime → Restart runtime")
    else:
        print("❌ Installation failed:", result.stderr[-800:])
else:
    print(f"⚠️  pyproject.toml not found at {DRIVE_PKG}")
    print("   To use the PyPI version instead, run Option B below.")

In [ ]:
# ── Option B: install from PyPI (PRODUCTION) ──────────────────────
# Uncomment the lines below ONLY if you do NOT have the package in Drive.
# After running, restart the runtime before continuing.

# %pip install -U speakerscribe --quiet
# print("✅ speakerscribe installed/updated from PyPI. Restart the runtime now.")

---
## Step 4 — Extended pre-flight check

This cell validates your entire environment before any heavy processing starts. It checks:

- ✅ **GPU availability** — confirms CUDA is accessible and reports VRAM
- ✅ **Disk space** — ensures enough space for model downloads and outputs
- ✅ **HuggingFace token** — verifies `HF_TOKEN` is set and readable
- ✅ **Audio files** — lists all media files found in `data/`, with size and duration
- ✅ **Chunking detection** — flags which files will be automatically split
- ✅ **Time estimate** — predicts total processing time based on model RTF

**Do not skip this step.** It catches environment issues (missing token, no GPU, empty folder) before you waste GPU time.

In [ ]:
import speakerscribe
from speakerscribe import (
    TranscriptionConfig, WorkspacePaths,
    preflight_check, get_audio_duration_seconds,
)

print(f"🔢 speakerscribe version: {speakerscribe.__version__}")

# Build the configuration object from the variables set in Step 2
config = TranscriptionConfig(
    model=MODEL,
    language=LANGUAGE,
    beam_size=5,
    enable_diarization=ENABLE_DIARIZATION,
    num_speakers=NUM_SPEAKERS,
    min_speakers=MIN_SPEAKERS if NUM_SPEAKERS is None else None,
    max_speakers=MAX_SPEAKERS if NUM_SPEAKERS is None else None,
    initial_prompt=INITIAL_PROMPT or None,
    vad_min_silence_ms=VAD_MIN_SILENCE_MS,
    gap_max_s_transcript=GAP_MAX_S,
    remove_fillers=REMOVE_FILLERS,
    long_audio_threshold_min=LONG_AUDIO_THRESHOLD_MIN,
    chunk_duration_min=CHUNK_DURATION_MIN,
    chunk_overlap_s=CHUNK_OVERLAP_S,
    force_reprocess=FORCE_REPROCESS,
    evaluate_quality=True,
    enable_runs_db=True,           # SQLite history for idempotency
    produce_unified_for_llm=True,  # Generate _full_for_llm.txt
)

# Initialize workspace paths and create required directories
paths = WorkspacePaths(workspace=WORKSPACE)
paths.create_directories()

# Scan the data/ folder for media files
media = paths.list_media_files()
if not media:
    raise FileNotFoundError(f"⚠️  No audio files found. Place them in: {paths.data}")

# Real-time factor estimates per model (how many seconds of audio per second of processing)
RTF_ESTIMATE = {"tiny": 12, "small": 8, "medium": 5, "large-v3-turbo": 3.5, "large-v3": 2.5}
factor = RTF_ESTIMATE.get(MODEL, 3.0)

print(f"\n📂 Files found in {paths.data}:")
total_min_audio = 0.0
total_mb = 0.0
n_chunked = 0

for i, f in enumerate(media, 1):
    size_mb = f.stat().st_size / 1e6
    total_mb += size_mb
    try:
        dur_min = get_audio_duration_seconds(f) / 60
    except Exception:
        dur_min = size_mb   # fallback estimate if ffprobe is unavailable
    total_min_audio += dur_min
    will_chunk = dur_min > LONG_AUDIO_THRESHOLD_MIN
    if will_chunk:
        n_chunked += 1
        n_chunks_est = int((dur_min // CHUNK_DURATION_MIN) + 1)
        chunk_info = f" -> ✂️  ~{n_chunks_est} chunks"
    else:
        chunk_info = ""
    has_glossary = "📚" if (f.with_suffix('.prompt.txt')).exists() else "  "
    print(f"   {has_glossary} {i}. {f.name}  ({size_mb:.1f} MB, {dur_min:.1f} min){chunk_info}")

# Time estimate
sec_process = (total_min_audio * 60) / factor
print(f"\n📊 Estimated processing time:")
print(f"   • Total audio    : {total_min_audio:.1f} min ({total_mb:.0f} MB)")
print(f"   • RTF for {MODEL}: ~{factor}x realtime")
print(f"   • Files to chunk : {n_chunked}")
print(f"   • Estimated time : ~{sec_process/60:.1f} min ({sec_process/3600:.2f} h)")
if sec_process > 3600 * 4:
    print(f"   ⚠️  >4h estimated. Consider splitting the batch or using 'small'.")

# Validate HuggingFace token (required for diarization)
if ENABLE_DIARIZATION:
    hf = config.resolve_hf_token()
    if not hf:
        raise EnvironmentError(
            "❌ HF_TOKEN not found. Set it in Colab Secrets (🔑 sidebar) "
            "as 'HF_TOKEN' and restart the runtime."
        )
    print(f"\n🔑 HF token detected: {hf[:6]}...{hf[-4:]}")

info = preflight_check(paths, config)
print(f"\n✅ Pre-flight OK — {info['n_files']} file(s) ready to process")
print(f"   📚 = file has a per-file glossary (.prompt.txt next to the audio)")

---
## Step 5 — Smoke test (using `tiny` model on the first file)

**Do not skip this.** The smoke test runs the full pipeline end-to-end in ~2 minutes using the lightweight `tiny` model, without diarization. It confirms that:

- The audio file is readable
- Whisper can transcribe it
- The output writing pipeline works correctly
- Language detection is functional

If the smoke test fails, you have a problem that would also cause the full run to fail — but you find out in 2 minutes instead of 2 hours.

> Set `RUN_SMOKE_TEST = False` to skip if you have already validated the pipeline recently.

In [ ]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    from speakerscribe.transcription import load_whisper_model, release_whisper_model
    from speakerscribe.pipeline import process_one
    import time

    # Smoke config: tiny model, no diarization, always reprocess
    smoke_config = TranscriptionConfig(
        model="tiny",
        language=LANGUAGE,
        enable_diarization=False,   # Skip diarization for speed
        force_reprocess=True,       # Always reprocess, ignore cache
        evaluate_quality=False,
        long_audio_threshold_min=0, # Never chunk during smoke test
        enable_runs_db=False,
    )

    first_file = media[0]
    print(f"🧪 Smoke test on: {first_file.name}")
    t0 = time.time()
    smoke_model = None
    try:
        smoke_model = load_whisper_model(smoke_config)
        smoke_result = process_one(first_file, paths, smoke_model, smoke_config)
        elapsed = time.time() - t0
        print(f"\n✅ Smoke test PASSED in {elapsed:.1f}s")
        print(f"   Status          : {smoke_result.get('status')}")
        print(f"   Word count      : {smoke_result.get('total_words', 0):,}")
        print(f"   Detected lang   : {smoke_result.get('language_detected', '-')}")
        print(f"   Stage timings   : {smoke_result.get('timings', {})}")
    except Exception as e:
        print(f"\n❌ Smoke test FAILED: {type(e).__name__}: {e}")
        raise
    finally:
        if smoke_model is not None:
            release_whisper_model(smoke_model)   # Free VRAM before loading the real model
else:
    print("⏩ Smoke test skipped (RUN_SMOKE_TEST=False)")

---
## Step 6 — Full batch processing

This is the main processing cell. It:

1. Loads the selected Whisper model into GPU memory (once)
2. Iterates over all audio files in `data/`
3. For each file:
   - Extracts audio to WAV if needed
   - Runs speaker diarization on the full audio (if enabled)
   - Splits long files into chunks (if threshold exceeded)
   - Transcribes with Whisper
   - Merges speaker labels + transcript segments
   - Writes all output files to `transcripts/` and `splits/`
   - Logs the result to SQLite for idempotency
4. Files already processed (same content hash) are automatically skipped

> **If the session expires mid-batch**: just restart and re-run Steps 1–6. The SQLite database (`_runs.db`) tracks which files completed successfully and skips them.

In [ ]:
%%time
from speakerscribe import process_batch

# Process all files — loads model once, then iterates
results = process_batch(paths, config)

# Summarize results
n_ok    = sum(1 for r in results if r.get("status") == "ok")
n_skip  = sum(1 for r in results if r.get("status") == "skipped")
n_err   = sum(1 for r in results if r.get("status") == "error")
n_words = sum(r.get("total_words", 0) for r in results if r.get("status") == "ok")

print(f"\n🎉 Batch complete")
print(f"   ✅ OK       : {n_ok}")
print(f"   ⏩ Skipped  : {n_skip}   (already processed — use FORCE_REPROCESS=True to redo)")
print(f"   ❌ Errors   : {n_err}")
print(f"   📝 Words    : {n_words:,}")
print(f"\n📁 Transcripts : {paths.transcripts}")
print(f"📁 Splits      : {paths.splits}")
print(f"💾 History DB  : {paths.db_path.name} (content-hash idempotency)")

---
## Step 7 — Build deliverables + consolidated summary

This cell collects the final outputs based on what you selected in Step 2 and copies them to `entregables/` — your clean delivery folder.

It also generates two metadata outputs:
- **`_resumen.md`**: a human-readable Markdown table summarizing all processed files (duration, language, speaker count, word count, RTF, quality flags, timing breakdown)
- **`_metadata/run_<timestamp>.json`**: full machine-readable run metadata for auditing and reproducibility

> ⚠️ Do **not** delete the `transcripts/` and `splits/` folders. They are the source of truth for idempotency — deleting them forces full reprocessing.

In [ ]:
%%time
import shutil
import json as _json
from pathlib import Path
from datetime import datetime
import pytz

# Set up the deliverables folder
deliverables = Path(WORKSPACE) / "entregables"
deliverables.mkdir(parents=True, exist_ok=True)
meta_dir = deliverables / "_metadata"
meta_dir.mkdir(exist_ok=True)

copied_files = []

for r in results:
    if r.get("status") != "ok":
        continue
    base = r.get("base_name")
    if not base:
        continue

    # Paths to all possible output files for this audio
    src_md   = paths.transcripts / f"{base}.transcript.md"
    src_json = paths.transcripts / f"{base}.json"
    src_srt  = paths.transcripts / f"{base}.srt"

    if COPY_TRANSCRIPT_MD and src_md.exists():
        shutil.copy2(src_md, deliverables / src_md.name)
        copied_files.append(src_md.name)

    if COPY_JSON and src_json.exists():
        shutil.copy2(src_json, deliverables / src_json.name)
        copied_files.append(src_json.name)

    if COPY_SRT and src_srt.exists():
        shutil.copy2(src_srt, deliverables / src_srt.name)
        copied_files.append(src_srt.name)

    if COPY_FULL_FOR_LLM:
        src_full = paths.splits / f"{base}_full_for_llm.txt"
        if src_full.exists():
            shutil.copy2(src_full, deliverables / src_full.name)
            copied_files.append(src_full.name)

    if COPY_SPLITS:
        for s in sorted(paths.splits.glob(f"{base}_*.txt")):
            if s.name.endswith("_full_for_llm.txt"):
                continue   # avoid duplicate copy
            shutil.copy2(s, deliverables / s.name)
            copied_files.append(s.name)

# ── Generate _resumen.md (human-readable batch summary) ──────────
ts_utc = datetime.now(pytz.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

lines = [
    f"# Batch Summary — {ts_utc}",
    "",
    f"- **speakerscribe version:** {speakerscribe.__version__}",
    f"- **Whisper model:** `{MODEL}`",
    f"- **Language:** {LANGUAGE or 'auto-detect'}",
    f"- **Diarization:** {'yes' if ENABLE_DIARIZATION else 'no'}",
    f"- **Chunking threshold:** {LONG_AUDIO_THRESHOLD_MIN} min",
    "",
    "## Results per file",
    "",
    "| # | Audio | Status | Duration | Language (conf) | Speakers | Words | RTF | Chunked | Quality flags |",
    "|---|---|---|---|---|---|---|---|---|---|",
]

for i, r in enumerate(results, 1):
    status = r.get("status", "?")
    audio  = r.get("audio_file", "?")
    if status == "ok":
        dur      = r.get("duration_minutes", 0)
        lang     = r.get("language_detected", "?")
        lang_p   = r.get("language_probability", 0)
        spk_summ = r.get("speakers_summary") or {}
        n_spk    = len(spk_summ)
        words    = r.get("total_words", 0)
        rtf      = r.get("real_time_factor", 0)
        chunked  = "✂️ yes" if r.get("chunked") else "no"
        flags    = r.get("quality_flags") or []
        flags_str = "; ".join(flags) if flags else "—"
        if len(flags_str) > 60:
            flags_str = flags_str[:57] + "..."
        lines.append(
            f"| {i} | `{audio}` | ✅ | {dur:.1f} min | {lang} ({lang_p:.0%}) | {n_spk} | {words:,} | {rtf:.1f}x | {chunked} | {flags_str} |"
        )
    elif status == "skipped":
        lines.append(f"| {i} | `{audio}` | ⏩ skipped | — | — | — | — | — | — | (already processed) |")
    elif status == "error":
        err = r.get("error", "unknown")[:80]
        lines.append(f"| {i} | `{audio}` | ❌ error | — | — | — | — | — | — | {err} |")

lines.extend([
    "",
    "## Stage timing breakdown (seconds)",
    "",
    "| # | Audio | extract_wav | diarization | split_audio | transcribe | transcript_md | total |",
    "|---|---|---|---|---|---|---|---|",
])
for i, r in enumerate(results, 1):
    if r.get("status") != "ok":
        continue
    t = r.get("timings", {})
    lines.append(
        f"| {i} | `{r.get('audio_file', '?')}` | "
        f"{t.get('extract_wav_s', '—')} | {t.get('diarization_s', '—')} | "
        f"{t.get('split_audio_s', '—')} | {t.get('transcribe_total_s', '—')} | "
        f"{t.get('transcript_md_s', '—')} | {t.get('total_s', '—')} |"
    )

summary_path = deliverables / "_resumen.md"
summary_path.write_text("\n".join(lines), encoding="utf-8")

# ── Write consolidated run metadata (JSON) ───────────────────────
ts_id = datetime.now(pytz.utc).strftime("%Y%m%d_%H%M%S")
run_metadata = {
    "timestamp_utc": ts_utc,
    "speakerscribe_version": speakerscribe.__version__,
    "config": config.model_dump(),
    "results_summary": {
        "ok": n_ok, "skipped": n_skip, "errors": n_err, "total_words": n_words,
    },
    "results": [
        {
            "audio_file"        : r.get("audio_file"),
            "status"            : r.get("status"),
            "base_name"         : r.get("base_name"),
            "duration_minutes"  : r.get("duration_minutes"),
            "language_detected" : r.get("language_detected"),
            "total_words"       : r.get("total_words"),
            "real_time_factor"  : r.get("real_time_factor"),
            "chunked"           : r.get("chunked"),
            "n_chunks"          : r.get("n_chunks"),
            "speakers_summary"  : r.get("speakers_summary"),
            "quality_ok"        : r.get("quality_ok"),
            "quality_flags"     : r.get("quality_flags"),
            "timings"           : r.get("timings"),
        }
        for r in results
    ],
}
meta_path = meta_dir / f"run_{ts_id}.json"
meta_path.write_text(_json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Final report ─────────────────────────────────────────────────
print(f"📦 Deliverables written to: {deliverables}")
print(f"   • Files copied      : {len(copied_files)}")
for c in copied_files[:10]:
    print(f"     - {c}")
if len(copied_files) > 10:
    print(f"     ... and {len(copied_files) - 10} more")
print(f"   • Batch summary     : {summary_path.name}")
print(f"   • Run metadata      : {meta_path.name}")
print(f"\n💡 Originals are intact in transcripts/ and splits/ — do not delete them (breaks idempotency).")

---
## Step 8 (optional) — Inspect a transcript JSON

Use this to quickly inspect the structured output of any processed file. The JSON contains the full segment-level data: start/end timestamps, speaker labels, word-level confidence, language detection probability, and pipeline telemetry.

Useful for: debugging alignment issues, verifying speaker detection quality, or extracting specific time ranges.

In [ ]:
from speakerscribe import inspect_json

json_files = sorted(paths.transcripts.glob("*.json"))
if json_files:
    print(f"📋 Inspecting: {json_files[0].name}\n")
    info = inspect_json(json_files[0])
else:
    print("⚠️  No JSON files found yet. Run Step 6 first.")

---
## Step 9 (optional) — Rename speakers

After reading the transcript and identifying who is speaking, use this cell to replace the anonymous labels (`SPEAKER_00`, `SPEAKER_01`, etc.) with real names.

**Workflow:**
1. Read the `.transcript.md` file in `entregables/`
2. Identify which `SPEAKER_XX` label corresponds to which person
3. Edit `SPEAKER_MAPPING` below with the correct names
4. Set `RUN_RENAME = True` and run the cell

The rename is applied to all output files for the specified `BASE_NAME` (`.md`, `.txt`, `.srt`, `_full_for_llm.txt`) and the updated deliverable is automatically copied.

In [ ]:
from speakerscribe import rename_speakers_in_outputs

# List all available base names to pick from
available_bases = sorted({
    f.stem.replace('.transcript', '')
    for f in paths.transcripts.glob('*.transcript.md')
})
print("📋 Available base names:")
for b in available_bases:
    print(f"   • {b}")

# ✏️ Edit these values:
BASE_NAME = available_bases[0] if available_bases else "your_audio_file_name"
SPEAKER_MAPPING = {
    "SPEAKER_00": "Alice",
    "SPEAKER_01": "Bob",
    # Add more mappings as needed
}

RUN_RENAME = False   # Change to True once you have edited SPEAKER_MAPPING

if RUN_RENAME:
    stats = rename_speakers_in_outputs(paths, BASE_NAME, SPEAKER_MAPPING)
    print(f"\n✅ Renamed speakers in {len(stats)} file(s):")
    for filename, count in stats.items():
        print(f"   {filename}: {count} replacement(s)")
    # Update the deliverable with the renamed version
    src = paths.transcripts / f"{BASE_NAME}.transcript.md"
    if src.exists() and COPY_TRANSCRIPT_MD:
        shutil.copy2(src, deliverables / src.name)
        print(f"   📦 Deliverable updated: {src.name}")
else:
    print("\n⏩ Edit SPEAKER_MAPPING and set RUN_RENAME=True to run.")

---
## Step 10 — Free GPU memory WITHOUT ending the session

Run this cell when you are done processing but want to keep the Colab session alive (e.g., to continue working in Python without the model in memory).

What it does:
- Deletes large objects (`results`, model variables, configs) from memory
- Calls Python garbage collector
- Clears CUDA cache and reports available VRAM

The session remains active — you can still run other cells.

In [ ]:
import gc
import torch

# Delete large objects from the global namespace
vars_to_delete = ["results", "smoke_result", "smoke_model", "config", "smoke_config"]
for var_name in vars_to_delete:
    if var_name in globals():
        del globals()[var_name]
        print(f"🧹 Freed: {var_name}")

# Python garbage collection
gc.collect()

# CUDA cache flush
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"\n💾 Free VRAM: {free_gb:.2f} GB")
else:
    print("\n⚠️  CUDA not available — no GPU memory to clear.")

print("✅ Memory freed. Session is still active.")

---
## Step 11 — Shut down runtime (releases GPU quota)

> ⚠️ **This cell will terminate your Colab session** if you uncomment the last line. All unsaved in-memory variables will be lost. Your Drive files are NOT affected.

Run this when you are completely done with the session to return the GPU to the Colab pool (important for staying within free-tier limits).

**Instructions**: uncomment the `runtime.unassign()` line and re-run the cell.

In [ ]:
# ⚠️  THIS CELL ENDS THE SESSION if you uncomment the last line.
from datetime import datetime
import pytz, gc, torch

print(f"🕐 Current time (UTC): {datetime.now(pytz.utc).strftime('%Y-%m-%d %H:%M:%S')}")

# Clean up remaining large variables
for var_name in list(globals().keys()):
    if var_name.startswith(("model", "results", "config", "paths")):
        try:
            del globals()[var_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n🛑 To shut down NOW, uncomment the line below and re-run this cell:")
print("    # from google.colab import runtime; runtime.unassign()")

# ─── UNCOMMENT THE LINE BELOW ONLY WHEN YOU ARE SURE ──────────────
# from google.colab import runtime; runtime.unassign()

---
## 📖 Quick reference

### Common scenarios

| Scenario | Solution |
|---|---|
| Session expired mid-batch | Re-run Steps 1–6. SQLite DB skips already-completed files automatically. |
| Changed `num_speakers` | Diarization cache auto-invalidates (param hash changed). No manual cleanup needed. |
| Added a new audio file | Only the new file is processed. Existing ones are skipped. |
| Want to force full reprocessing | Set `FORCE_REPROCESS = True` in Step 2. |
| Smoke test failed | Check GPU runtime, verify audio file is not corrupted, confirm ffprobe is installed. |

---

### Per-file glossary

Place a plain text file named `mymeeting.prompt.txt` next to `mymeeting.mp4`:

```
ProColombia, Bancoldex, GIC, Analytics Committee, John Smith, Q3 Report
```

speakerscribe auto-detects it and uses it as `initial_prompt` **only for that audio file**, overriding the global `INITIAL_PROMPT`.

---

### Output files produced per audio

| Folder | File | Contents |
|---|---|---|
| `transcripts/` | `<base>.txt` | Plain text with `[SPEAKER_XX]` labels |
| `transcripts/` | `<base>.srt` | Subtitle file with timestamps |
| `transcripts/` | `<base>.json` | Full metadata + segments (audit trail) |
| `transcripts/` | `<base>.transcript.md` | **Human-readable Markdown** ⭐ |
| `splits/` | `<base>_N.txt` | Context-window-sized chunks for short-context LLMs |
| `splits/` | `<base>_full_for_llm.txt` | Full transcript for long-context LLMs ⭐ |

---

### How to generate a meeting summary from the transcript

1. Ensure `COPY_FULL_FOR_LLM = True` in Step 2.
2. Open the `*_full_for_llm.txt` file from `entregables/`.
3. Paste it into Claude or GPT with a prompt like:

> *"Based on the following meeting transcript, generate a structured summary including: attendees, key decisions, action items with owners, and follow-up dates. Transcript: [paste here]"*

---

### Resources

- 📦 [speakerscribe on PyPI](https://pypi.org/project/speakerscribe/)
- 📁 [GitHub repository](https://github.com/EnriqueForero/speakerscribe)
- 🐛 [Report an issue](https://github.com/EnriqueForero/speakerscribe/issues)
- 🤗 [faster-whisper](https://github.com/SYSTRAN/faster-whisper)
- 🤗 [pyannote.audio](https://github.com/pyannote/pyannote-audio)